In [24]:
# ── Librerías ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import janitor
import sqlalchemy as sa
import os
from pathlib import Path
from dotenv import load_dotenv

%matplotlib inline

# ── Opciones de visualización ─────────────────────────────────────────
# Desactivar notación científica
pd.set_option('display.float_format', '{:.3f}'.format)
np.set_printoptions(suppress=True)

# Ver todas las columnas al imprimir un DataFrame
pd.set_option('display.max_columns', None)

# ── Rutas del proyecto ────────────────────────────────────────────────
# Se resuelven desde la ubicación del notebook, así funcionan
# igual en cualquier máquina
RAIZ = Path.cwd().parent

ORIGINALES   = RAIZ / '02_datos' / '01_Originales'
VALIDACION   = RAIZ / '02_datos' / '02_Validacion'
ENTRENAMIENTO = RAIZ / '02_datos' / '03_Entrenamiento'
CACHES       = RAIZ / '02_datos' / '04_Caches'
MODELOS      = RAIZ / '05_modelos'
RESULTADOS   = RAIZ / '06_resultados'

# ── Variables de entorno ──────────────────────────────────────────────
load_dotenv(RAIZ / '.env')

print('✅ Entorno listo')
print(f'   Raíz del proyecto: {RAIZ}')

✅ Entorno listo
   Raíz del proyecto: c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Mis Casos(Importante)\PROYECTO 5 - FORECASTING RETAIL\02_FORECASTING


In [25]:
type(RAIZ)

pathlib.WindowsPath

In [26]:
#CARGA
df = pd.read_pickle(ENTRENAMIENTO / 'train.pkl')


## VISION GENERAL Y TIPOS DE DATOS

In [27]:
df.info()
df.shape

<class 'pandas.DataFrame'>
Index: 21280 entries, 2013-01-01 to 2015-11-30
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   state_id      21280 non-null  str    
 1   store_id      21280 non-null  str    
 2   dept_id       21280 non-null  str    
 3   cat_id        21280 non-null  str    
 4   item_id       21280 non-null  str    
 5   wm_yr_wk      21280 non-null  int64  
 6   dia_ind       21280 non-null  str    
 7   ventas        21280 non-null  int64  
 8   sell_price    21209 non-null  float64
 9   year          21280 non-null  int64  
 10  month         21280 non-null  int64  
 11  wday          21280 non-null  int64  
 12  weekday       21280 non-null  str    
 13  event_name_1  1700 non-null   str    
 14  event_type_1  1700 non-null   str    
 15  event_name_2  60 non-null     str    
 16  event_type_2  60 non-null     str    
dtypes: float64(1), int64(5), str(11)
memory usage: 2.9+ MB


(21280, 17)

- year, month y wday deberias ser categorias, porque son mas para cruces que agregaciones y metricas
  aunque venga como numerico para mas para categorica y sus dimenesiones

In [28]:
df['year'] = df['year'].astype('str')
df['month'] = df['month'].astype('str')
df['wday'] = df['wday'].astype('str')

## VALORES UNICOS

In [29]:
#identificacion
df.nunique().sort_values()

state_id           1
dept_id            1
cat_id             1
store_id           2
event_type_2       2
event_name_2       3
year               3
event_type_1       4
weekday            7
wday               7
item_id           10
month             12
sell_price        23
event_name_1      30
wm_yr_wk         153
ventas           301
dia_ind         1064
dtype: int64

In [30]:
#correcion
df.drop(columns=['state_id','cat_id','dept_id'], inplace=True)

## DUPLICADOS


In [31]:
df.duplicated().sum()

np.int64(0)

## SEPARAR NUMERICAS Y CATEGORICAS

In [77]:
#categoricas
cat = df.select_dtypes(include=['str']).copy()
con = df.select_dtypes(include=['number']).copy()

## CATEGORICAS

### Nulos, identificacion y tratamiento

In [33]:
cat.isna().sum().sort_values(ascending=False)

event_name_2    21220
event_type_2    21220
event_name_1    19580
event_type_1    19580
dia_ind             0
store_id            0
item_id             0
weekday             0
wday                0
month               0
year                0
dtype: int64

In [34]:
#Aparecen 60 NewYear, 60 SuperBowl, 60 Christmas entre otros porque son 10 items en 2 stores durante 3 años (2013-2015)
cat['event_name_1'].value_counts(dropna=False)

event_name_1
NaN                    19580
NewYear                   60
OrthodoxChristmas         60
MartinLutherKingDay       60
SuperBowl                 60
LentStart                 60
ValentinesDay             60
PresidentsDay             60
LentWeek2                 60
Purim End                 60
StPatricksDay             60
Easter                    60
Pesach End                60
Mother's day              60
MemorialDay               60
NBAFinalsStart            60
NBAFinalsEnd              60
IndependenceDay           60
Ramadan starts            60
Eid al-Fitr               60
LaborDay                  60
ColumbusDay               60
EidAlAdha                 60
Halloween                 60
VeteransDay               60
Thanksgiving              60
OrthodoxEaster            40
Father's day              40
Chanukah End              40
Christmas                 40
Cinco De Mayo             40
Name: count, dtype: int64

In [35]:
cat.event_name_2.value_counts(dropna=False)

event_name_2
NaN               21220
Cinco De Mayo        20
OrthodoxEaster       20
Father's day         20
Name: count, dtype: int64

### Correcion

In [78]:
cat.drop(columns = ['event_name_2','event_type_2'], inplace=True)

In [79]:
#imputamos los nulos de event_name_1 y event_type_1 por Sin_evento
cat.fillna({'event_name_1': 'Sin_evento', 'event_type_1': 'Sin_evento'}, inplace=True)

,store_id,item_id,dia_ind,year,month,wday,weekday,event_name_1,event_type_1
date,,,,,,,,,
2013-01-01,CA_3,FOODS_3_090,d_704,2013,1,4,Tuesday,NewYear,National
2013-01-01,CA_3,FOODS_3_120,d_704,2013,1,4,Tuesday,NewYear,National
2013-01-01,CA_3,FOODS_3_202,d_704,2013,1,4,Tuesday,NewYear,National
2013-01-01,CA_3,FOODS_3_252,d_704,2013,1,4,Tuesday,NewYear,National
2013-01-01,CA_3,FOODS_3_288,d_704,2013,1,4,Tuesday,NewYear,National
...,...,...,...,...,...,...,...,...,...
2015-11-30,CA_4,FOODS_3_329,d_1767,2015,11,3,Monday,Sin_evento,Sin_evento
2015-11-30,CA_4,FOODS_3_555,d_1767,2015,11,3,Monday,Sin_evento,Sin_evento
2015-11-30,CA_4,FOODS_3_586,d_1767,2015,11,3,Monday,Sin_evento,Sin_evento


### Atipicos

Categorias poco frecuentes

In [38]:
for variable in cat:
    print(variable + '\n')
    print(cat[variable].value_counts(dropna=False))
    print('\n')

store_id

store_id
CA_3    10640
CA_4    10640
Name: count, dtype: int64


item_id

item_id
FOODS_3_090    2128
FOODS_3_120    2128
FOODS_3_202    2128
FOODS_3_252    2128
FOODS_3_288    2128
FOODS_3_329    2128
FOODS_3_555    2128
FOODS_3_586    2128
FOODS_3_587    2128
FOODS_3_714    2128
Name: count, dtype: int64


dia_ind

dia_ind
d_704     20
d_705     20
d_706     20
d_707     20
d_708     20
          ..
d_1763    20
d_1764    20
d_1765    20
d_1766    20
d_1767    20
Name: count, Length: 1064, dtype: int64


year

year
2013    7300
2014    7300
2015    6680
Name: count, dtype: int64


month

month
1     1860
3     1860
5     1860
7     1860
8     1860
10    1860
4     1800
6     1800
9     1800
11    1800
2     1680
12    1240
Name: count, dtype: int64


wday

wday
4    3040
5    3040
6    3040
7    3040
1    3040
2    3040
3    3040
Name: count, dtype: int64


weekday

weekday
Tuesday      3040
Wednesday    3040
Thursday     3040
Friday       3040
Saturday     3040
Sunday     

## NUMERICAS

### Estadisticos

In [39]:
con.describe().T

,count,mean,std,min,25%,50%,75%,max
wm_yr_wk,21280.000,11415.023,82.875,11249.000,11335.000,11420.000,11506.000,11544.000
ventas,21280.000,28.875,38.988,0.000,6.000,18.000,37.000,763.000
sell_price,21209.000,2.397,1.237,1.000,1.500,1.580,2.980,4.980


### Nulos

In [40]:
con.isna().sum().sort_values(ascending=False)

sell_price    71
wm_yr_wk       0
ventas         0
dtype: int64

sell_price, analisis para encontrar algo

In [41]:
#esta sección analiza los valores nulos en la columna 'sell_price' agrupados por 'item_id'
df.groupby('item_id')['sell_price'].apply(lambda x: x.isna().sum()).sort_values(ascending=False)

item_id
FOODS_3_288    71
FOODS_3_090     0
FOODS_3_120     0
FOODS_3_202     0
FOODS_3_252     0
FOODS_3_329     0
FOODS_3_555     0
FOODS_3_586     0
FOODS_3_587     0
FOODS_3_714     0
Name: sell_price, dtype: int64

In [48]:
analisis_sell_price_nulos = df.loc[df['sell_price'].isna()]
analisis_sell_price_nulos['month'].value_counts()

month
1    62
2     9
Name: count, dtype: int64

El item FOODS_3_288 tiene valores nulos en la columna 'sell_price' durante los meses enero y febrero de 2013, lo que podría indicar un problema temporal en la disponibilidad de datos para este producto en esos meses, o que no existía un precio registrado para este producto en ese periodo.

In [50]:
analisis_sell_price_nulos['ventas'].value_counts()

ventas
0    71
Name: count, dtype: int64

In [56]:
df_prueba_foods_3 = df.loc[(df['item_id']=='FOODS_3_288') & (df['year'] == '2014') & (df['month'] == '4')]
df_prueba_foods_3['item_id'].value_counts()

item_id
FOODS_3_288    60
Name: count, dtype: int64

### Atipicos

Por desviacion Tipica

In [63]:
num_desv_tip = 4

In [64]:
def atipicos_desv_tip(variable, num_desv_tip=4):
    #sacamos los nulos por ahora
    variable = variable.dropna()
    #calculamos los limites
    media = np.mean(variable)
    desviacion = np.std(variable)
    umbral = num_desv_tip * desviacion
    limite_superior = media + umbral
    limite_inferior = media - umbral
    #encontramos los indices de los que estan afuera de los limites
    indices = [indice for indice,valor in variable.items() if valor > limite_superior or valor < limite_inferior]
    return indices


In [65]:
def conteo_atipicos(df, variable, num_desv_tip=4):
    atipicos = atipicos_desv_tip(df[variable], num_desv_tip=num_desv_tip)
    return (df.loc[atipicos,variable].value_counts())

In [66]:
var_atipicos_dt = ['ventas','sell_price']

In [67]:
for variable in var_atipicos_dt:
    print(f"Conteo de atípicos para {variable}:")
    print(conteo_atipicos(con, variable))

Conteo de atípicos para ventas:
ventas
0      518
1      103
5      100
10      95
7       92
      ... 
158      1
139      1
103      1
107      1
323      1
Name: count, Length: 247, dtype: int64
Conteo de atípicos para sell_price:
Series([], Name: count, dtype: int64)


## UNION - CON SALVEDAD DEL VOLVER A SEPARAR

In [80]:
df_2 = pd.concat([con, cat], axis=1)

In [81]:
# ── Truncado de registros pre-lanzamiento ────────────────────────────
# Se eliminan las 71 filas de FOODS_3_288 sin precio (ene–feb 2013).
# OJO: NO usar reset_index(drop=True) aquí. Esa llamada destruye el
# índice temporal 'date' heredado de train.pkl y lo sustituye por un
# RangeIndex 0..n, perdiendo la referencia temporal de todo el tablón.
df_2 = df_2.dropna(subset=['sell_price'])

# Salvaguarda: concat puede dejar el nombre del índice a None si los
# índices de origen no coinciden en nombre.
df_2.index.name = 'date'


### Corrección — conservación del índice temporal `date`

| Artefacto | Índice antes | Índice después |
|---|---|---|
| `train.pkl` (origen) | `Index` llamado `date` (2013-01-01 → 2015-11-30) | sin cambios |
| `df_2` / `train_resultado_calidad.pkl` | `RangeIndex` 0 → 21.208 ❌ | `Index` llamado `date` ✅ |

**Causa raíz:** el truncado se hacía con `dropna(...).reset_index(drop=True)`. El `drop=True` descarta el índice anterior y genera un `RangeIndex` nuevo, de modo que la fecha desaparecía del tablón guardado.

**Por qué no ocurría antes:** `select_dtypes()` y `pd.concat(axis=1)` **sí** conservan el índice; el único punto del flujo que lo destruía era ese `reset_index(drop=True)`.

**Consideración importante:** el índice `date` **no es único** (1.064 fechas distintas repetidas en 20 series = 2.128 registros por día antes del truncado). Por eso `df.loc['2013-01-01']` devuelve 20 filas. Si más adelante se necesita una clave única, la candidata es `['date', 'store_id', 'item_id']`.


In [82]:
#SEPARACION NUEVAMENTE, SE COLOCARA EN EL INFORME DE CALIDAD DE DATOS
cat = df_2.select_dtypes(include=['str']).copy()
con = df_2.select_dtypes(include=['number']).copy()

In [87]:
# ── Verificación del índice temporal ─────────────────────────────────
# select_dtypes() solo filtra columnas: el índice viaja intacto a cat y con.
vistas = {
    'df (origen)': df,
    'df_2 (tablón limpio)': df_2,
    'cat (categóricas)': cat,
    'con (numéricas)': con,
}

filas_vistas = []
for nombre, tabla in vistas.items():
    filas_vistas.append({
        'vista': nombre,
        'filas': tabla.shape[0],
        'columnas': tabla.shape[1],
        'tipo_indice': type(tabla.index).__name__,
        'nombre_indice': tabla.index.name,
        'primera_fecha': str(tabla.index.min()),
        'ultima_fecha': str(tabla.index.max()),
    })

df_vistas = pd.DataFrame(filas_vistas).set_index('vista')
print('df_2, cat y con comparten el mismo índice:', cat.index.equals(con.index) and cat.index.equals(df_2.index))
df_vistas


df_2, cat y con comparten el mismo índice: True


,filas,columnas,tipo_indice,nombre_indice,primera_fecha,ultima_fecha
vista,,,,,,
df (origen),21280,14,Index,date,2013-01-01,2015-11-30
df_2 (tablón limpio),21209,12,Index,date,2013-01-01,2015-11-30
cat (categóricas),21209,9,Index,date,2013-01-01,2015-11-30
con (numéricas),21209,3,Index,date,2013-01-01,2015-11-30


## GUARDAR DATASET EL CATEGORICO Y EL NUMERICO 
Se guardan como archivos .pkl el train completo que viene de df_2, cat, con en la carpeta de 03_Entrenamiento

In [85]:
df_2.to_pickle("../02_datos/03_Entrenamiento/train_resultado_calidad.pkl")
cat.to_pickle("../02_datos/03_Entrenamiento/cat_resultado_calidad.pkl")
con.to_pickle("../02_datos/03_Entrenamiento/num_resultado_calidad.pkl")

### Nota Metodológica: Tratamiento de Registros Inactivos en `sell_price`

**Diagnóstico:**
Se detectaron 71 valores nulos en `sell_price`, focalizados en el producto `FOODS_3_288` durante enero y los primeros días de febrero de 2013, coincidiendo con ventas en cero. El análisis confirma que se trata del período previo a la introducción del producto en el catálogo comercial de las tiendas, y no de una omisión aleatoria en la captura de precios.

**Decisión metodológica aplicada:**
Se descartó la imputación mediante modas semanales o globales para prevenir:
1. **Fuga de datos (*Data Leakage*):** Evitar trasladar precios promedio futuros (2014-2015) hacia las semanas iniciales de 2013.
2. **Distorsión en la relación precio-demanda:** Evitar enseñarle al modelo de Machine Learning que existió demanda cero frente a un precio ficticio en góndola.

**Acción ejecutada:**
Se unificaron previamente las matrices numéricas y categóricas (`pd.concat([num, cat], axis=1)`) para resguardar la integridad y consistencia de los índices. Posteriormente, se filtraron las 71 observaciones donde `sell_price` era nulo. 

Con este procedimiento, la serie temporal de `FOODS_3_288` inicia formalmente a partir de su lanzamiento al mercado, manteniendo el histórico íntegro para los restantes 9 productos sin introducir sesgos artificiales.

In [86]:
# ── Verificación del guardado ────────────────────────────────────────
# Lee los tres artefactos persistidos y confirma que conservan el
# índice temporal 'date' (no un RangeIndex).
artefactos = {
    'train_resultado_calidad.pkl': ENTRENAMIENTO / 'train_resultado_calidad.pkl',
    'cat_resultado_calidad.pkl': ENTRENAMIENTO / 'cat_resultado_calidad.pkl',
    'num_resultado_calidad.pkl': ENTRENAMIENTO / 'num_resultado_calidad.pkl',
}

filas_guardado = []
for nombre, ruta in artefactos.items():
    tabla = pd.read_pickle(ruta)
    idx = tabla.index
    filas_guardado.append({
        'artefacto': nombre,
        'filas': tabla.shape[0],
        'columnas': tabla.shape[1],
        'tipo_indice': type(idx).__name__,
        'nombre_indice': idx.name,
        'primera_fecha': str(idx.min()),
        'ultima_fecha': str(idx.max()),
    })

df_guardado = pd.DataFrame(filas_guardado).set_index('artefacto')
df_guardado


,filas,columnas,tipo_indice,nombre_indice,primera_fecha,ultima_fecha
artefacto,,,,,,
train_resultado_calidad.pkl,21209,12,Index,date,2013-01-01,2015-11-30
cat_resultado_calidad.pkl,21209,9,Index,date,2013-01-01,2015-11-30
num_resultado_calidad.pkl,21209,3,Index,date,2013-01-01,2015-11-30


In [88]:
# ── Estructura final del tablón limpio ───────────────────────────────
df_2.info()


<class 'pandas.DataFrame'>
Index: 21209 entries, 2013-01-01 to 2015-11-30
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   wm_yr_wk      21209 non-null  int64  
 1   ventas        21209 non-null  int64  
 2   sell_price    21209 non-null  float64
 3   store_id      21209 non-null  str    
 4   item_id       21209 non-null  str    
 5   dia_ind       21209 non-null  str    
 6   year          21209 non-null  str    
 7   month         21209 non-null  str    
 8   wday          21209 non-null  str    
 9   weekday       21209 non-null  str    
 10  event_name_1  21209 non-null  str    
 11  event_type_1  21209 non-null  str    
dtypes: float64(1), int64(2), str(9)
memory usage: 2.1+ MB
